In [ ]:
import pandas as pd
import time

In [ ]:
data = pd.read_csv('../data/raw/portfolio - portfolio.csv')

In [ ]:
data.head()

In [ ]:
data.info()

### Existence Checks

In [ ]:
MUST_HAVE_COLUMNS = ["Portfolio", "Transaction Date", "Transaction Type", "Ticker", "Quantity", "Transaction Price", "Fee"]

missing_columns = []
for col in MUST_HAVE_COLUMNS:
    if col not in data.columns:
        missing_columns.append(col)

if len(missing_columns) > 0:
    raise ValueError(f"The following columns are missing from the data: {missing_columns}")

data = data[MUST_HAVE_COLUMNS]

In [ ]:
data.info()

### Data Type Checks

In [ ]:
MUST_HAVE_DATA_TYPES = {
    "Portfolio": lambda x: isinstance(x, str) or isinstance(x, int),                        # string or integer
    "Transaction Date": lambda x: 
        (pd.to_datetime(x, format='%Y%m%d', errors='coerce') is not pd.NaT) or 
        (pd.to_datetime(x, format='%Y-%m-%d', errors='coerce') is not pd.NaT),              # format should match YYYY-MM-DD or YYYYMMDD
    "Transaction Type": lambda x: isinstance(x, str),                                       # string
    "Ticker": lambda x: isinstance(x, str),                                                 # string
    "Quantity": lambda x: isinstance(x, int),                                               # integet
    "Transaction Price": lambda x: (isinstance(x, float) or isinstance(x, int)),            # float or integer
    "Fee": lambda x: (isinstance(x, float) or isinstance(x, int))                           # float or integer
}

data_types_violated = []
for col in MUST_HAVE_DATA_TYPES:
    if not all(data[col].apply(MUST_HAVE_DATA_TYPES[col])):
        data_types_violated.append(col)

if data_types_violated:
    raise ValueError(f"Some values in columns {data_types_violated} do not meet the required data types.")

# convert data types to the required 
MUST_HAVE_DATA_TYPES = {
    "Portfolio": str, 
    "Transaction Date": str,
    "Transaction Type": str,
    "Ticker": str,
    "Quantity": int,
    "Transaction Price": float,
    "Fee": float
}
data = data.astype(MUST_HAVE_DATA_TYPES)

# specific handling of date column
if any(data["Transaction Date"].apply(lambda x: (pd.to_datetime(x, format='%Y%m%d', errors='coerce') is not pd.NaT))):
    data["Transaction Date"] = pd.to_datetime(data["Transaction Date"], format='%Y%m%d', errors='coerce')
else:
    data["Transaction Date"] = pd.to_datetime(data["Transaction Date"], format='%Y-%m-%d', errors='coerce')

# specific handling of asset column
data["Ticker"] = data["Ticker"].str.strip().str.upper()

In [ ]:
data.info()

### Content Validity Checks

In [ ]:
MUST_HAVE_DATA_RULES = {
    "Portfolio": lambda x: len(x) > 0,                                                  # at least one character long
    "Transaction Date": lambda x: x <= pd.to_datetime(time.ctime()),                    # date before current time
    "Transaction Type": lambda x: x in ["BUY", "SELL"],                                 # available values: BUY, SELL
    "Ticker": lambda x: len(x) > 0,                                                     # at least one character long
    "Quantity": lambda x: x > 0,                                                        # positive number
    "Transaction Price": lambda x: x >= 0,                                              # non-negative number
    "Fee": lambda x: x >= 0                                                             # non-negative number
}

rules_violated = []
for col in MUST_HAVE_DATA_RULES:
    if not all(data[col].apply(MUST_HAVE_DATA_RULES[col])):
        rules_violated.append(col)

if rules_violated:
    raise ValueError(f"Some values in columns {rules_violated} do not meet the required rules.")

In [ ]:
data.head()